In [ ]:
%pip install pandas matplotlib seaborn geopandas pycountry numpy

: 

In [ ]:
# biblioteki
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import csv
import re
import geopandas as gpd
import pycountry
import numpy as np

sns.set_theme(style="whitegrid", context="notebook")


##  Tworzenie datasetów

In [ ]:
# dataset OpenAlex
df_ap = pd.read_csv('datasets/OpenAlex.csv')
df_ap.head()

In [ ]:
print(df_ap.columns)

In [ ]:
#dateset Scopus

df_sc = pd.read_csv('datasets/scopus_export_Mar 19-2026.csv')
df_sc.head()

In [ ]:
print(df_sc.columns)

In [ ]:
# dataset web of science
params = {
    'sep': '\t',
    'quoting': csv.QUOTE_NONE,
    'encoding': 'utf-8', 
    'index_col': False,
    'on_bad_lines': 'warn'
}

files = [
    r'datasets/WebOfScience1-1000.csv',
    r'datasets/WebOfScience1001-2000.csv',
    r'datasets/WebOfScience2001-2100.csv'
]

df_list = []
for f in files:
    temp_df = pd.read_csv(f, **params)
    df_list.append(temp_df)

df_wos = pd.concat(df_list, axis=0, ignore_index=True)


mapping = {
    'FN': 'File Name',
    'VR': 'Version Number',
    'PT': 'Publication Type',
    'AU': 'Authors',
    'AF': 'Author Full Name',
    'BA': 'Book Authors',
    'BF': 'Book Authors Full Name',
    'CA': 'Group Authors',
    'GP': 'Book Group Authors',
    'BE': 'Editors',
    'TI': 'Document Title',
    'SO': 'Publication Name',
    'SE': 'Book Series Title',
    'BS': 'Book Series Subtitle',
    'LA': 'Language',
    'DT': 'Document Type',
    'CT': 'Conference Title',
    'CY': 'Conference Date',
    'CL': 'Conference Location',
    'SP': 'Conference Sponsors',
    'HO': 'Conference Host',
    'DE': 'Author Keywords',
    'ID': 'Keywords Plus®',
    'AB': 'Abstract',
    'C1': 'Author Address',
    'RP': 'Reprint Address',
    'EM': 'E-mail Address',
    'RI': 'ResearcherID Number',
    'OI': 'ORCID Identifier (Open Researcher and Contributor ID)',
    'FU': 'Funding Agency and Grant Number',
    'FX': 'Funding Text',
    'CR': 'Cited References',
    'NR': 'Cited Reference Count',
    'TC': 'Web of Science Core Collection Times Cited Count',
    'Z9': 'Total Times Cited Count',
    'U1': 'Usage Count (Last 180 Days)',
    'U2': 'Usage Count (Since 2013)',
    'PU': 'Publisher',
    'PI': 'Publisher City',
    'PA': 'Publisher Address',
    'SN': 'International Standard Serial Number (ISSN)',
    'EI': 'Electronic International Standard Serial Number (eISSN)',
    'BN': 'International Standard Book Number (ISBN)',
    'J9': '29-Character Source Abbreviation',
    'JI': 'ISO Source Abbreviation',
    'PD': 'Publication Date',
    'PY': 'Year Published',
    'VL': 'Volume',
    'IS': 'Issue',
    'SI': 'Special Issue',
    'PN': 'Part Number',
    'SU': 'Supplement',
    'MA': 'Meeting Abstract',
    'BP': 'Beginning Page',
    'EP': 'Ending Page',
    'AR': 'Article Number',
    'DI': 'Digital Object Identifier (DOI)',
    'D2': 'Book Digital Object Identifier (DOI)',
    'PG': 'Page Count',
    'P2': 'Chapter Count (Book Citation Index)',
    'WC': 'Web of Science Categories',
    'SC': 'Research Areas',
    'GA': 'Document Delivery Number',
    'UT': 'Accession Number',
    'PM': 'PubMed ID',
    'ER': 'End of Record',
    'EF': 'End of File'}


df_wos = df_wos.rename(columns=mapping)

print(f"The file has {len(df_wos)} records.")

In [ ]:
df_wos.head()

# Świat

## Chronologiczny rozkład publikacji

In [ ]:
# open alex 
yearly_counts_op = df_ap['publication_year'].value_counts().sort_index()



In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ostatni rok w danych
last_year = yearly_counts_op.index.max()
# Dane pełne, but exclude 2026
yc_full = yearly_counts_op[yearly_counts_op.index < last_year]
# Dane niepełne (ostatni rok)
yc_partial = yearly_counts_op[yearly_counts_op.index == last_year]

fig, ax = plt.subplots(figsize=(10, 5))

# Linia dla danych pełnych
sns.lineplot(
    x=yc_full.index, y=yc_full.values,
    marker="o", linewidth=2, markersize=5,
    label="Dane pełne", ax=ax
)

# Linia dla danych niepełnych (ostatni rok)
sns.lineplot(
    x=yc_partial.index, y=yc_partial.values,
    marker="o", markersize=8, linewidth=2, linestyle="--",
    label=f"Dane niepełne ({last_year})", ax=ax
)

# --- Etykiety wartości co 2 lata ---
offset = (max(yc_full.values) * 0.03)  # 3% wysokości — wyraźnie wyżej

for x, y in zip(yc_full.index, yc_full.values):
    if x % 2 == 0 or x == 2025:  # Etykiety co 2 lata lub dla ostatniego roku
        ax.text(
            x, y + offset,
            str(y), ha='center', va='bottom',
            fontsize=10
        )

# --- Etykieta dla ostatniego roku ---
for x, y in zip(yc_partial.index, yc_partial.values):
    ax.text(
        x, y + offset,
        str(y), ha='center', va='bottom',
        fontsize=10, color=sns.color_palette()[1]
    )

ax.set(title='Chronologiczny rozkład publikacji (OpenAlex)',
       xlabel='Rok publikacji', ylabel='Liczba publikacji')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
print(last_year)

In [ ]:
sns.regplot(x=yc_full.index, y=yc_full.values, data=yc_full, order=3)


In [ ]:
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

last_year = yearly_counts_op.index.max()
yc_full = yearly_counts_op[yearly_counts_op.index < last_year]
yc_partial = yearly_counts_op[yearly_counts_op.index == last_year]

# Okres wzrostu wykładniczego
yc_growth = yc_full[yc_full.index >= 2008]

fig, ax = plt.subplots(figsize=(10, 5))

# Linia dla danych pełnych
sns.lineplot(
    x=yc_full.index, y=yc_full.values,
    marker="o", linewidth=2, markersize=5,
    label="Dane pełne", ax=ax
)

# Linia dla danych niepełnych (ostatni rok)
sns.lineplot(
    x=yc_partial.index, y=yc_partial.values,
    marker="o", markersize=8, linewidth=2, linestyle="--",
    label=f"Dane niepełne ({last_year})", ax=ax
)

# --- Trend wykładniczy od 2008 (dopasowanie ręczne) ---
x_g = yc_growth.index.values.astype(float)
y_g = yc_growth.values.astype(float)

# Regresja liniowa na log(y): log(y) = a*x + b  =>  y = exp(b) * exp(a*x)
a, b = np.polyfit(x_g, np.log(y_g), 1)

x_trend = np.linspace(x_g.min(), x_g.max(), 200)
y_trend = np.exp(a * x_trend + b)

ax.plot(
    x_trend, y_trend,
    linewidth=2, linestyle=":", alpha=0.8,
    color=sns.color_palette()[2],
    label="Trend wykładniczy (od 2008)"
)

# --- Etykiety wartości co 2 lata ---
offset = (max(yc_full.values) * 0.03)
for x, y in zip(yc_full.index, yc_full.values):
    if x % 2 == 0 or x == 2025:
        ax.text(x, y + offset, str(y), ha='center', va='bottom', fontsize=10)

# --- Etykieta dla ostatniego roku ---
for x, y in zip(yc_partial.index, yc_partial.values):
    ax.text(x, y + offset, str(y), ha='center', va='bottom',
            fontsize=10, color=sns.color_palette()[1])

ax.set(title='Chronologiczny rozkład publikacji (OpenAlex)',
       xlabel='Rok publikacji', ylabel='Liczba publikacji')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
#sns.regplot(x=yc_full.index, y=yc_full.values, data=yc_full,lowess = True)

In [ ]:
yc_full.head()

In [ ]:
yc_full['log_y'] = np.log1p(df_ap['publication_year'].value_counts().sort_index())


df= df_ap['publication_year'].value_counts().sort_index()



In [ ]:

df_ap['log_y'] = np.log1p(df_ap['publication_year'])
df_ap['log_y'] 

In [ ]:
# ax = sns.regplot(x=yc_full.index, y=yc_full.values, order=3)
# ax.set(title='Regresja 3. stopnia (OpenAlex)', xlabel='Rok publikacji', ylabel='Liczba publikacji')
# plt.show()

In [ ]:
# # Dane Scopus
# yearly_counts_sc = df_sc['Year'].value_counts().sort_index()

# # Ostatni rok
# last_year_sc = yearly_counts_sc.index.max()

# # Dane pełne i niepełne
# sc_full = yearly_counts_sc[yearly_counts_sc.index < last_year_sc]
# sc_partial = yearly_counts_sc[yearly_counts_sc.index == last_year_sc]

# fig, ax = plt.subplots(figsize=(10, 5))

# # Linia dla danych pełnych
# sns.lineplot(
#     x=sc_full.index, y=sc_full.values,
#     marker="o", linewidth=2, markersize=5,
#     label="Dane pełne", ax=ax
# )

# # Linia dla danych niepełnych
# sns.lineplot(
#     x=sc_partial.index, y=sc_partial.values,
#     marker="o", markersize=8, linewidth=2, linestyle="--",
#     label=f"Dane niepełne ({last_year_sc})", ax=ax
# )

# # Offset etykiet (3% wysokości)
# offset_sc = max(sc_full.values) * 0.03

# # --- Etykiety wartości co 2 lata ---
# for x, y in zip(sc_full.index, sc_full.values):
#     if x % 2 == 0 or x == 2025:  # Etykiety co 2 lata lub dla ostatniego roku
#         ax.text(
#             x, y + offset_sc,
#             str(y), ha='center', va='bottom',
#             fontsize=10
#         )

# # Etykieta dla ostatniego roku
# for x, y in zip(sc_partial.index, sc_partial.values):
#     ax.text(
#         x, y + offset_sc,
#         str(y), ha='center', va='bottom',
#         fontsize=10, color=sns.color_palette()[1]
#     )

# # Informacja o niepełnym roku
# fig.text(
#     0.5, -0.02,
#     "Dane za rok 2026 obejmują okres do 19 marca 2026.",
#     ha='center', fontsize=10, color='gray'
# )

# ax.set(title='Chronologiczny rozkład publikacji (Scopus)',
#        xlabel='Rok publikacji', ylabel='Liczba publikacji')
# ax.legend()
# fig.tight_layout()
# plt.show()

In [ ]:
# sns.regplot(x=sc_full.index, y=sc_full.values, data=sc_full, order=2)

In [ ]:
# Dane Scopus
yearly_counts_sc = df_sc['Year'].value_counts().sort_index()

# Ostatni rok
last_year_sc = yearly_counts_sc.index.max()

# Dane pełne i niepełne
sc_full = yearly_counts_sc[yearly_counts_sc.index < last_year_sc]
sc_partial = yearly_counts_sc[yearly_counts_sc.index == last_year_sc]

# --- Dopasowanie prawa potęgowego: y = A * t^k, t = rok - 2009 ---
t0 = sc_full.index.min() - 1                      # rok bazowy: 2009
t = sc_full.index.values.astype(float) - t0       # 1, 2, ..., 16
counts = sc_full.values.astype(float)

k, logA = np.polyfit(np.log(t), np.log(counts), 1)
A = np.exp(logA)

x_trend = np.linspace(sc_full.index.min(), sc_full.index.max(), 200)
y_trend = A * (x_trend - t0) ** k

# --- Wykres ---
fig, ax = plt.subplots(figsize=(10, 5))

# Linia dla danych pełnych
sns.lineplot(
    x=sc_full.index, y=sc_full.values,
    marker="o", linewidth=2, markersize=5,
    label="Dane pełne", ax=ax
)

# Linia dla danych niepełnych
sns.lineplot(
    x=sc_partial.index, y=sc_partial.values,
    marker="o", markersize=8, linewidth=2, linestyle="--",
    label=f"Dane niepełne ({last_year_sc})", ax=ax
)

# Krzywa trendu potęgowego
ax.plot(
    x_trend, y_trend,
    linestyle=":", linewidth=1.5, color="gray", alpha=0.8,
    label=f"Trend potęgowy ($y \\sim t^{{{k:.2f}}}$)"
)

# Offset etykiet (3% wysokości)
offset_sc = max(sc_full.values) * 0.03

# --- Etykiety wartości co 2 lata ---
for x, y in zip(sc_full.index, sc_full.values):
    if x % 2 == 0 or x == 2025:
        ax.text(
            x, y + offset_sc,
            str(y), ha='center', va='bottom',
            fontsize=10
        )

# Etykieta dla ostatniego roku
for x, y in zip(sc_partial.index, sc_partial.values):
    ax.text(
        x, y + offset_sc,
        str(y), ha='center', va='bottom',
        fontsize=10, color=sns.color_palette()[1]
    )

# Informacja o niepełnym roku
fig.text(
    0.5, -0.02,
    "Dane za rok 2026 obejmują okres do 19 marca 2026.",
    ha='center', fontsize=10, color='gray'
)

ax.set(title='Chronologiczny rozkład publikacji (Scopus)',
       xlabel='Rok publikacji', ylabel='Liczba publikacji')
ax.legend()
fig.tight_layout()
plt.show()

# --- Statystyki dopasowania ---
log_pred = k * np.log(t) + logA
ss_res = np.sum((np.log(counts) - log_pred) ** 2)
ss_tot = np.sum((np.log(counts) - np.log(counts).mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"Wykładnik k = {k:.2f}, A = {A:.2f}, R² (log-log) = {r2:.3f}")

In [ ]:
# # Dane Web of Science
# yearly_counts_wof = df_wos['Year Published'].value_counts().sort_index()

# # Ostatni rok
# last_year_wof = yearly_counts_wof.index.max()

# # Dane pełne i niepełne
# wof_full = yearly_counts_wof[yearly_counts_wof.index < last_year_wof]
# wof_partial = yearly_counts_wof[yearly_counts_wof.index == last_year_wof]

# fig, ax = plt.subplots(figsize=(10, 5))

# # Linia dla danych pełnych
# sns.lineplot(
#     x=wof_full.index, y=wof_full.values,
#     marker="o", linewidth=2, markersize=5,
#     label="Dane pełne", ax=ax
# )

# # Linia dla danych niepełnych
# sns.lineplot(
#     x=wof_partial.index, y=wof_partial.values,
#     marker="o", markersize=8, linewidth=2, linestyle="--",
#     label=f"Dane niepełne ({last_year_wof})", ax=ax
# )

# # Offset etykiet (3% wysokości)
# offset_wof = max(wof_full.values) * 0.03

# # Etykiety co 2 lata
# for x, y in zip(wof_full.index, wof_full.values):
#     if x % 2 == 0 or x == 2025:  # Etykiety co 2 lata lub dla ostatniego roku
#         ax.text(
#             x, y + offset_wof,
#             str(y), ha='center', va='bottom', fontsize=10
#         )

# # Etykieta dla ostatniego roku
# for x, y in zip(wof_partial.index, wof_partial.values):
#     ax.text(
#         x, y + offset_wof,
#         str(y), ha='center', va='bottom',
#         fontsize=10, color=sns.color_palette()[1]
#     )

# # Informacja o niepełnym roku
# fig.text(
#     0.5, -0.02,
#     "Dane za rok 2026 obejmują okres do 19 marca 2026.",
#     ha='center', fontsize=10, color='gray'
# )

# ax.set(title='Chronologiczny rozkład publikacji (Web of Science)',
#        xlabel='Rok publikacji', ylabel='Liczba publikacji')
# ax.legend()
# fig.tight_layout()
# plt.show()

In [ ]:
# Dane Web of Science
yearly_counts_wof = df_wos['Year Published'].value_counts().sort_index()

# Ostatni rok
last_year_wof = yearly_counts_wof.index.max()

# Dane pełne i niepełne
wof_full = yearly_counts_wof[yearly_counts_wof.index < last_year_wof]
wof_partial = yearly_counts_wof[yearly_counts_wof.index == last_year_wof]

# --- Dopasowanie prawa potęgowego: y = A * t^k ---
t0_wof = wof_full.index.min() - 1                       # rok bazowy
t_wof = wof_full.index.values.astype(float) - t0_wof    # 1, 2, ...
counts_wof = wof_full.values.astype(float)

k_wof, logA_wof = np.polyfit(np.log(t_wof), np.log(counts_wof), 1)
A_wof = np.exp(logA_wof)

x_trend_wof = np.linspace(wof_full.index.min(), wof_full.index.max(), 200)
y_trend_wof = A_wof * (x_trend_wof - t0_wof) ** k_wof

# --- Wykres ---
fig, ax = plt.subplots(figsize=(10, 5))

# Linia dla danych pełnych
sns.lineplot(
    x=wof_full.index, y=wof_full.values,
    marker="o", linewidth=2, markersize=5,
    label="Dane pełne", ax=ax
)

# Linia dla danych niepełnych
sns.lineplot(
    x=wof_partial.index, y=wof_partial.values,
    marker="o", markersize=8, linewidth=2, linestyle="--",
    label=f"Dane niepełne ({last_year_wof})", ax=ax
)

# Krzywa trendu potęgowego
ax.plot(
    x_trend_wof, y_trend_wof,
    linestyle=":", linewidth=1.5, color="gray", alpha=0.8,
    label=f"Trend potęgowy ($y \\sim t^{{{k_wof:.2f}}}$)"
)

# Offset etykiet (3% wysokości)
offset_wof = max(wof_full.values) * 0.03

# Etykiety co 2 lata
for x, y in zip(wof_full.index, wof_full.values):
    if x % 2 == 0 or x == 2025:
        ax.text(
            x, y + offset_wof,
            str(y), ha='center', va='bottom', fontsize=10
        )

# Etykieta dla ostatniego roku
for x, y in zip(wof_partial.index, wof_partial.values):
    ax.text(
        x, y + offset_wof,
        str(y), ha='center', va='bottom',
        fontsize=10, color=sns.color_palette()[1]
    )

# Informacja o niepełnym roku
fig.text(
    0.5, -0.02,
    "Dane za rok 2026 obejmują okres do 19 marca 2026.",
    ha='center', fontsize=10, color='gray'
)

ax.set(title='Chronologiczny rozkład publikacji (Web of Science)',
       xlabel='Rok publikacji', ylabel='Liczba publikacji')
ax.legend()
fig.tight_layout()
plt.show()

# --- Statystyki dopasowania ---
log_pred_wof = k_wof * np.log(t_wof) + logA_wof
ss_res = np.sum((np.log(counts_wof) - log_pred_wof) ** 2)
ss_tot = np.sum((np.log(counts_wof) - np.log(counts_wof).mean()) ** 2)
r2_wof = 1 - ss_res / ss_tot

print(f"WoS: wykładnik k = {k_wof:.2f}, A = {A_wof:.2f}, R² (log-log) = {r2_wof:.3f}")

In [ ]:
# sns.regplot(x=wof_full.index, y=wof_full.values, data=wof_full, order=2)

## Rozkład geograficzny publikacji, recenzji, artykułów naukowych dotyczących tematu zarejestrowanych w bazach danych 


In [ ]:
# Open Alex
df_ap_coun = df_ap['authorships.countries']
df_ap_coun = df_ap_coun.dropna()
df_ap_coun.head()


In [ ]:
# funkcja do rozdzielenia ciągu znaków po seperatorze '|'
def clean_and_get_unique_countries(text):
    codes = str(text).split('|')
    
    unique_codes = set([code.strip() for code in codes if code.strip() != ''])
    
    return list(unique_codes)

In [ ]:
df_ap_coun = df_ap_coun.apply(clean_and_get_unique_countries)
print(df_ap_coun.head())

In [ ]:
country_counts = df_ap_coun.explode().value_counts()
print(country_counts.head(10))
# add human-readable names of countries

In [ ]:
top = country_counts.head(10).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, hue=top.index,
            palette="viridis", legend=False, ax=ax)
ax.set(title='Rozkład geograficzny publikacji (OpenAlex)',
       xlabel='Liczba publikacji (unikalny udział kraju w artykule)',
       ylabel='Kod kraju (ISO Alpha-2)')
fig.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
import pycountry
import matplotlib.pyplot as plt

def alpha2_to_alpha3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except Exception:
        return None

country_df = country_counts.reset_index()
country_df.columns = ['country_code', 'count']
country_df['iso_a3'] = country_df['country_code'].apply(alpha2_to_alpha3)
country_df = country_df.dropna(subset=['iso_a3'])

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)[['ISO_A3', 'geometry']].rename(columns={'ISO_A3': 'iso_a3'})
world_merged = world.merge(country_df, on='iso_a3', how='left')

fig, ax = plt.subplots(figsize=(14, 7))
world_merged.plot(
    column='count',
    ax=ax,
    cmap='Blues',
    legend=True,
    missing_kwds={'color': 'lightgrey'},
    legend_kwds={'label': 'Liczba publikacji', 'shrink': 0.6}
)
ax.set_title('Rozkład geograficzny publikacji (OpenAlex)')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Scopus
df_sc['Affiliations'][0]

In [ ]:
#Funkcja wyciągająca nazwy krajów z  Scopus
def get_countries(affiliation_text):
    affiliations = str(affiliation_text).split(';')

    unique_countries = set()
    
    for aff in affiliations:
        parts = aff.split(',')
        if len(parts) > 0:
            country = parts[-1].strip() 
            if country and not country.isdigit():
                unique_countries.add(country)
                
    return list(unique_countries)

In [ ]:
df_sc_coun = df_sc['Affiliations']
df_sc_coun = df_sc_coun.dropna()
df_sc_coun.head()


In [ ]:
df_sc_coun = df_sc_coun.apply(get_countries)
df_sc_coun.head()

In [ ]:
scopus_country_counts = df_sc_coun.explode().dropna().map(str.strip).value_counts()
scopus_country_counts = scopus_country_counts.rename(
    index=lambda x: 'USA' if 'United States' in str(x) else x
)

print(scopus_country_counts.head(10))

print(scopus_country_counts.head(10))

In [ ]:
top = scopus_country_counts.head(10).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, hue=top.index,
            palette="viridis", legend=False, ax=ax)
ax.set(title='Rozkład geograficzny publikacji (Scopus)',
       xlabel='Liczba publikacji',
       ylabel='Kraj')
fig.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
import pycountry
import matplotlib.pyplot as plt

def name_to_alpha3(name):
    try:
        return pycountry.countries.search_fuzzy(name)[0].alpha_3
    except Exception:
        return None

scopus_df = scopus_country_counts.reset_index()
scopus_df.columns = ['country_name', 'count']
scopus_df['iso_a3'] = scopus_df['country_name'].apply(name_to_alpha3)
scopus_df = scopus_df.dropna(subset=['iso_a3'])

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)[['ISO_A3', 'geometry']].rename(columns={'ISO_A3': 'iso_a3'})
world_merged = world.merge(scopus_df, on='iso_a3', how='left')

fig, ax = plt.subplots(figsize=(14, 7))
world_merged.plot(
    column='count',
    ax=ax,
    cmap='Blues',
    legend=True,
    missing_kwds={'color': 'lightgrey'},
    legend_kwds={'label': 'Liczba publikacji', 'shrink': 0.6}
)
ax.set_title('Rozkład geograficzny publikacji (Scopus)')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
df_wos.columns

In [ ]:
# Web of Science

df_wos_coun = df_wos['Author Address']
df_wos_coun = df_wos_coun.dropna()
df_wos_coun


In [ ]:

def get_wos_countries(address_text):
    if pd.isna(address_text):
        return []
        
    text = str(address_text)
    text_no_authors = re.sub(r'\[.*?\]', '', text)
    affiliations = text_no_authors.split(';')
    
    unique_countries = set()
    for aff in affiliations:
        aff = aff.strip()
        if not aff:
            continue
            
        aff_cleaned = aff.rstrip('.')
        words = aff_cleaned.split()
        if words:
            country = words[-1].strip()
            
            if 'Peoples R China' in aff:
                country = 'China'
                
            if country and not country.isdigit():

                unique_countries.add(country)
                
    return list(unique_countries)


In [ ]:
df_wos_coun['Country'] = df_wos_coun.apply(get_wos_countries)
df_wos_coun['Country'].head()

In [ ]:
wos_country_counts = df_wos_coun['Country'].explode().value_counts()
print("Top 10 państw w bazie Web of Science:")
print(wos_country_counts.head(10))

In [ ]:
top = wos_country_counts.head(10).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, hue=top.index,
            palette="viridis", legend=False, ax=ax)
ax.set(title='Rozkład geograficzny publikacji (Web of Science)',
       xlabel='Liczba publikacji',
       ylabel='Kraj')
fig.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
import pycountry
import matplotlib.pyplot as plt

def name_to_alpha3(name):
    try:
        return pycountry.countries.search_fuzzy(name)[0].alpha_3
    except Exception:
        return None

scopus_df = wos_country_counts.reset_index()
scopus_df.columns = ['country_name', 'count']
scopus_df['iso_a3'] = scopus_df['country_name'].apply(name_to_alpha3)
scopus_df = scopus_df.dropna(subset=['iso_a3'])

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)[['ISO_A3', 'geometry']].rename(columns={'ISO_A3': 'iso_a3'})
world_merged = world.merge(scopus_df, on='iso_a3', how='left')

fig, ax = plt.subplots(figsize=(14, 7))
world_merged.plot(
    column='count',
    ax=ax,
    cmap='Blues',
    legend=True,
    missing_kwds={'color': 'lightgrey'},
    legend_kwds={'label': 'Liczba publikacji', 'shrink': 0.6}
)
ax.set_title('Rozkład geograficzny publikacji (Web of Science)')
ax.axis('off')
plt.tight_layout()
plt.show()

## Rozkład tematu według dyscyplin naukowych


In [ ]:
# Open Alex
df_op_areas = df_ap['primary_topic.display_name'].dropna().value_counts()

print("Top 10 dziedzin w OpenAlex:")
print(df_op_areas.head(10))

In [ ]:
top = df_op_areas.head(10).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, hue=top.index,
            palette="crest", legend=False, ax=ax)
ax.set(title='Rozkład publikacji według dyscyplin (Open Alex)',
       xlabel='Liczba publikacji',
       ylabel='Obszar badawczy (primary_topic.display_name)')
fig.tight_layout()
plt.show()

In [ ]:
df_sc.columns 

In [ ]:
# Scopus
df_sc_areas = df_sc['Source title'].dropna().value_counts()# Find Publisher and find their domain
print("Top 10 dziedzin w Scopus:")
print(df_sc_areas.head(10))

In [ ]:
top = df_sc_areas.head(10).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, hue=top.index,
            palette="crest", legend=False, ax=ax)
ax.set(title='Rozkład publikacji według dyscyplin (Scopus)',
       xlabel='Liczba publikacji',
       ylabel='Obszar badawczy (Source title)')
fig.tight_layout()
plt.show()

In [ ]:
df_wos.columns # same as SCOPUS, use Publisher column

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Scopus → dyscypliny ASJC z oficjalnej Scopus Source List (czerwiec 2026)
# Plik: ext_list_Jun_2026.xlsx  (Scopus → Sources → Download Scopus Source List)
# ══════════════════════════════════════════════════════════════════════
XLSX = 'ext_list_Jun_2026.xlsx'   # dostosuj ścieżkę

# --- normalizatory ---
def norm_issn(s):
    if pd.isna(s): return None
    s = re.sub(r'[^0-9X]', '', str(s).upper())
    return s.zfill(8) if 0 < len(s) <= 8 else None

def norm_title(s):
    if pd.isna(s): return None
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def norm_isbn(s):
    if pd.isna(s): return None
    return re.sub(r'[^0-9X]', '', str(s).upper()) or None

def split_codes(s):
    if pd.isna(s): return []
    return [c.strip() for c in str(s).split(';') if c.strip()]

# --- słownik ASJC: kod → opis, prefiks (setki) → nazwa dziedziny ---
asjc = pd.read_excel(XLSX, sheet_name='ASJC Classification Codes',
                     skiprows=9, usecols=[0, 1], names=['code', 'desc'], dtype=str)
asjc_desc, group_name, current = {}, {}, None
for _, r in asjc.iterrows():
    code, desc = r['code'], r['desc']
    if pd.isna(code) and pd.notna(desc):          # nagłówek grupy
        current = desc.strip()
    if pd.notna(code) and str(code).strip().isdigit():
        code = str(code).strip()
        asjc_desc[code] = (desc or code).strip()
        if code.endswith('00') and pd.notna(desc):
            name = desc.strip()
            name = name[len('General '):] if name.startswith('General ') else name
            group_name[code[:2]] = name          # np. '17' → 'Computer Science'

def field(code):                                  # kod → ogólna dziedzina (27 grup)
    return group_name.get(str(code)[:2], 'Inne/Nieznane')

# --- mapy: ISSN/EISSN → kody, tytuł → kody, ISBN → kody ---
issn2codes, title2codes, isbn2codes, conftitle2codes = {}, {}, {}, {}

for sheet, acol in [('Scopus Sources Jun. 2026',
                     'All Science Journal Classification Codes (ASJC)'),
                    ('Serial Conf. Proc. with Profile',
                     'All Science Journal Classification Codes (ASJC)')]:
    d = pd.read_excel(XLSX, sheet_name=sheet, dtype=str)
    d['cl'] = d[acol].map(split_codes)
    for _, r in d.iterrows():
        for col in ('ISSN', 'EISSN'):
            k = norm_issn(r.get(col))
            if k: issn2codes.setdefault(k, r['cl'])
        t = norm_title(r['Source Title'])
        if t: title2codes.setdefault(t, r['cl'])

conf = pd.read_excel(XLSX, sheet_name='All Conf. Proceedings Jun. 2026', dtype=str)
conf['cl'] = conf['ASJC'].map(split_codes)
for _, r in conf.iterrows():
    k = norm_isbn(r['ISBN'])
    if k: isbn2codes.setdefault(k, r['cl'])
    t = norm_title(r['Source Title'])
    if t: conftitle2codes.setdefault(t, r['cl'])

print(f"Mapy: ISSN {len(issn2codes)} | tytuł-czasop. {len(title2codes)} | "
      f"ISBN {len(isbn2codes)} | tytuł-konf. {len(conftitle2codes)}")

In [ ]:
# Web of Science
df_wos_areas = df_wos['Research Areas']
df_wos_areas.dropna()
df_wos_areas.head()

In [ ]:
# funkcja na podzielenie dziedzin 
def get_wos_areas(area_text):
    areas = str(area_text).split(';')
    return [area.strip() for area in areas if area.strip() != '']

In [ ]:
df_wos_areas = df_wos_areas.apply(get_wos_areas)
wos_areas_counts = df_wos_areas.explode().value_counts()
print("Top 10 dziedzin w Web of Science: ")
print(wos_areas_counts.head(10))

In [ ]:
top = wos_areas_counts.head(10).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, hue=top.index,
            palette="crest", legend=False, ax=ax)
ax.set(title='Rozkład publikacji według dyscyplin (Web of Science)',
       xlabel='Liczba publikacji',
       ylabel='Obszar badawczy (Research Area)')
fig.tight_layout()
plt.show()

## Rozkład wg języków publikacji


In [ ]:

oa_lang_counts = df_ap['language'].dropna().value_counts().head(10)
sc_lang_counts = df_sc['Language of Original Document'].dropna().value_counts().head(10)
wos_lang_counts = df_wos['Language'].dropna().value_counts().head(10)


In [ ]:

print("Top 5 języków (Open Alex)")
print(oa_lang_counts.head(5))
print("Top 5 języków (Scopus)")
print(sc_lang_counts.head(5))
print("Top 5 języków (Web of Science)")
print(wos_lang_counts.head(5))

## Liczba publikacji

In [ ]:
total_openalex = len(df_ap)
total_scopus = len(df_sc)
total_wos = len(df_wos)

In [ ]:

print("PODSUMOWANIE CAŁKOWITEJ LICZBY PUBLIKACJI ")
print(f"OpenAlex:        {total_openalex}")
print(f"Scopus:          {total_scopus}")
print(f"Web of Science:  {total_wos}")


## liczba współpracujących autorów

In [ ]:
# open alex
df_openalex_authors = df_ap['authorships.author.id'].dropna()
all_authors_openalex = df_openalex_authors.str.split(';').explode().str.strip()
openalex_unique_authors_count = all_authors_openalex.nunique()
openalex_avg_authors_per_paper = len(all_authors_openalex) / len(df_openalex_authors)
print(openalex_avg_authors_per_paper)

In [ ]:
# scopus
df_scopus_authors = df_sc['Author(s) ID'].dropna()
all_authors_scopus = df_scopus_authors.str.split(';').explode().str.strip()
scopus_unique_authors_count = all_authors_scopus.nunique()
scopus_avg_authors_per_paper = len(all_authors_scopus) / len(df_scopus_authors)
print


In [ ]:
# Web of science
df_wos_authors = df_wos['ORCID Identifier (Open Researcher and Contributor ID)'].dropna()
all_authors_wos = df_wos_authors.str.split(';').explode().str.strip()
wos_unique_authors_count = all_authors_wos.nunique()
wos_avg_authors_per_paper = len(all_authors_wos) / len(df_wos_authors)

In [ ]:
print("Open ALex ")
print(f"Całkowita liczba współpracujących autorów (unikalnych): {openalex_unique_authors_count}")
print(f"Średnia liczba autorów na jedną publikację:            {openalex_avg_authors_per_paper:.2f}")

print("Scopus")
print(f"Całkowita liczba współpracujących autorów (unikalnych): {scopus_unique_authors_count}")
print(f"Średnia liczba autorów na jedną publikację:            {scopus_avg_authors_per_paper:.2f}")

print("Web of Science")
print(f"Całkowita liczba współpracujących autorów (unikalnych): {wos_unique_authors_count}")
print(f"Średnia liczba autorów na jedną publikację:            {wos_avg_authors_per_paper:.2f}")





In [ ]:
print("Top 5 najbardzij płodnych autorów (Open Alex)")
print(all_authors_openalex.value_counts().head(5))

print("Top 5 najbardzij płodnych autorów (Scopus)")
print(all_authors_scopus.value_counts().head(5))
print("Top 5 najbardzij płodnych autorów (Web of Science)")
print(all_authors_wos.value_counts().head(5))

## Wskaźnik produktywności na aktywne lata publikacji (Productivity per Active Year)

In [ ]:

def calculate_productivity(name, years_series):
    if len(years_series) == 0:
        return None
    
    total_pub = len(years_series)
    active_years = years_series.nunique()
    
    productivity = total_pub / active_years
    
    return {
        "Base": name,
        "Total Publications": total_pub,
        "Active Years": active_years,
        "Productivity (Pub/Year)": round(productivity, 2)
    }



In [ ]:
scopus_years = pd.to_numeric(df_sc['Year'], errors='coerce').dropna().astype(int)

wos_years = pd.to_numeric(df_wos['Year Published'], errors='coerce').dropna().astype(int)

oa_years = pd.to_numeric(df_ap['publication_year'], errors='coerce').dropna().astype(int)


In [ ]:
productivit = []
productivit.append(calculate_productivity("Open Alex",oa_years))
productivit.append(calculate_productivity("Scopus",scopus_years))
productivit.append(calculate_productivity("Web of Science",wos_years))



In [ ]:
print(productivit)

In [ ]:

df_productivity = pd.DataFrame([s for s in productivit if s is not None])

print("Productivity per Active Year")
print(df_productivity.to_string(index=False))

## całkowita liczba cytowań (Total Citation Count)

In [ ]:
# OPEn Alex
oa_citations = pd.to_numeric(df_ap['cited_by_count'], errors='coerce').fillna(0)

total_cit_oa = oa_citations.sum()
avg_cit_oa = oa_citations.mean()


In [ ]:
# scopus

scopus_citations = pd.to_numeric(df_sc['Cited by'], errors='coerce').fillna(0)

total_cit_scopus = scopus_citations.sum()
avg_cit_scopus = scopus_citations.mean()


In [ ]:
# web of Science
wos_citations = pd.to_numeric(df_wos['Total Times Cited Count'], errors='coerce').fillna(0)

total_cit_wos = wos_citations.sum()
avg_cit_wos = wos_citations.mean()

In [ ]:
print(f"Analiza cytowań")
print(f"Open Alex - Suma cytowań:{int(total_cit_oa)}")
print(f"Open Alex - Średnia na artykuł: {int(avg_cit_oa)}")

print("-" * 30)
print(f"Scopus - Suma cytowań: {int(total_cit_scopus)}")
print(f"Scopus - Średnia na artykuł: {avg_cit_scopus:.2f}")
print("-" * 30)
print(f"WoS - Suma cytowań:{int(total_cit_wos)}")
print(f"WoS - Średnia na artykuł:{avg_cit_wos:.2f}")
